# VELNOR Qwen3-8B QLoRA — E2 (Free GPU)

This notebook runs the repository's canonical Qwen3-8B 4-bit NF4 QLoRA trainer against the current Velnor curriculum. It is adapter fine-tuning, not foundation-model pretraining.

**Execution policy:** free CUDA GPU only; at least 14 GiB total VRAM and 12 GiB free VRAM; no paid runtime; no automatic model substitution. The 1.5B baseline remains unchanged. Training artifacts stay in Google Drive and are not committed to Git.

E2 fixes the E1 notebook's cross-model adapter reuse, global configuration leakage, duplicate GitHub push/PR workflows, force pushes, and unsafe artifact handling. The previous run's validation loss is historical metadata only: checkpoint selection starts from the resumed adapter's loss on this run's exact held-out split.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Free CUDA GPU required. CPU training is disabled; no training started.')
props = torch.cuda.get_device_properties(0)
free_bytes, _ = torch.cuda.mem_get_info(0)
total_gib = props.total_memory / 1024**3
free_gib = free_bytes / 1024**3
print({'gpu': props.name, 'total_vram_gib': round(total_gib, 2), 'free_vram_gib': round(free_gib, 2), 'cuda': torch.version.cuda})
if total_gib < 14 or free_gib < 12:
    raise RuntimeError('Qwen3-8B requires >=14 GiB total and >=12 GiB free VRAM. No training started.')

In [ ]:
from pathlib import Path
import subprocess
REPO = Path('/content/velnor')
BRANCH = 'feat/qwen3-8b-qlora'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/vectolith/velnor.git', str(REPO)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
    branch = subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip()
    if remote.rstrip('.git') != 'https://github.com/vectolith/velnor' or branch != BRANCH:
        raise RuntimeError(f'Existing checkout mismatch: remote={remote}, branch={branch}. Use a fresh Colab runtime; E2 will not delete or reset files.')
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository:', remote if REPO.exists() else 'vectolith/velnor', '| branch:', BRANCH, '| commit:', COMMIT)

In [ ]:
%cd /content/velnor
!python -m pip install -q -r training/requirements-qwen3-qlora.txt
import torch
assert torch.cuda.is_available(), 'CUDA became unavailable during dependency installation.'
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
!python -m py_compile training/train_qwen3_8b_qlora.py

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess
RUNS = Path('/content/drive/MyDrive/VELNOR-training-runs')
RUNS.mkdir(parents=True, exist_ok=True)
# E1's explicitly selected original Qwen3-8B adapter. Do not silently fall back to another model or a fresh run.
PRIOR = RUNS / 'qwen3-8b-20261112T123354Z' / 'outputs' / 'adapter'
if not (PRIOR / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'Required prior Qwen3 adapter not found: {PRIOR}. Restore the original run from Drive before continuing.')
if not any((PRIOR / name).is_file() for name in ('adapter_model.safetensors', 'adapter_model.bin')):
    raise FileNotFoundError(f'Prior adapter weights are missing: {PRIOR}')
print('Resume adapter:', PRIOR)

In [ ]:
from datetime import datetime, timezone
RUN_ID = 'qwen3-8b-e2-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = RUNS / RUN_ID
OUT_DIR = RUN_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=False)
os.environ['VELNOR_QWEN3_OUTPUT'] = str(OUT_DIR)
os.environ['VELNOR_QWEN3_RESUME'] = str(PRIOR)
os.environ['VELNOR_QWEN3_PREV_BEST_LOSS'] = '1.19551'  # historical reference only; not used for selection
os.environ['VELNOR_QWEN3_MAX_SEQ_LEN'] = '256'
os.environ['VELNOR_QWEN3_EPOCHS'] = '1'
os.environ['VELNOR_QWEN3_GRAD_ACCUM'] = '8'
os.environ['VELNOR_QWEN3_PATIENCE'] = '1'
os.environ['VELNOR_QWEN3_LR'] = '1e-4'
print('Run ID:', RUN_ID, '| output:', OUT_DIR)

In [ ]:
import subprocess
log_path = OUT_DIR / 'training.log'
command = ['python', '-u', 'training/train_qwen3_8b_qlora.py']
with log_path.open('w', encoding='utf-8') as log:
    result = subprocess.run(command, cwd='/content/velnor', stdout=log, stderr=subprocess.STDOUT, text=True)
if result.returncode != 0:
    tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
    raise RuntimeError('Training process failed with exit code ' + str(result.returncode) + '\n' + '\n'.join(tail))
required = [OUT_DIR / 'adapter' / 'adapter_config.json', OUT_DIR / 'training_report.json', OUT_DIR / 'SHA256SUMS']
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise RuntimeError('Training exited successfully but required artifacts are missing: ' + ', '.join(missing))
print('Training process exit code:', result.returncode, '| artifacts verified.')

In [ ]:
import json
report = json.loads((OUT_DIR / 'training_report.json').read_text(encoding='utf-8'))
assert report.get('status') == 'completed', report.get('status')
assert report.get('base_model') == 'Qwen/Qwen3-8B', report.get('base_model')
assert report.get('dataset', {}).get('train_examples') == 288, report.get('dataset')
assert report.get('dataset', {}).get('heldout_examples') == 30, report.get('dataset')
print(json.dumps({
    'run': report.get('run'),
    'base_model': report.get('base_model'),
    'dataset': report.get('dataset'),
    'validation': report.get('validation'),
    'rubric': report.get('rubric'),
    'promotion_gate': report.get('promotion_gate'),
    'elapsed_seconds': report.get('elapsed_seconds'),
}, indent=2))

In [ ]:
from pathlib import Path
import hashlib
manifest = OUT_DIR / 'SHA256SUMS'
lines = manifest.read_text(encoding='utf-8').splitlines()
for line in lines:
    expected, relative = line.split('  ', 1)
    artifact = OUT_DIR / relative
    actual = hashlib.sha256(artifact.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Checksum mismatch: {relative}')
print('Verified SHA-256 entries:', len(lines))
print('Run directory is preserved in Google Drive:', RUN_DIR)
print('Model weights are not uploaded to GitHub. Review promotion_gate before any deployment.')

## Completion and release boundary

A successful notebook execution proves that this training job completed and its artifacts passed structural and checksum checks. It does **not** prove production readiness. Promote the adapter only if `promotion_gate.eligible_for_human_review` is true, then perform human review and broader regression testing. If the gate is false, retain the previous adapter and treat this run as an experiment.